## Data Loading

In [385]:
%%time

import kplex_dev
import data_splits
import warnings
import pandas as pd
import numpy as np
from openpyxl import Workbook

# create once, pass around multiple times for writing metrics
wb = Workbook()

warnings.filterwarnings('ignore')

timeout_replacement = 1800  # max timeout replacement
excel_path = "kplex_ML_data.xlsx"


CPU times: total: 0 ns
Wall time: 44 ms


In [386]:
# Loading and preprocessing data
cat_df, dfs = kplex_dev.load_excel_data(excel_path, timeout_replacement)
kplex_dev.process_kcore_list_to_statistics(cat_df)

# Selecting weight scheme
epsilon = 0.001 # need to set !!!
scheme = kplex_dev.W_TIME_MIN_X_W_LOG_TIME_RATIO # Change this and analyse!!

# Assigning weights
kplex_dev.assign_weights(cat_df, scheme, epsilon)

# kplex_dev.export_sorted(cat_df)

# Data reformatting
cat_df = kplex_dev.post_process_data(cat_df)
cat_df

,dataset,k_value,n,m,d_avg,d_max,core_max,core_min,shell_num,shell_avgSize,shell_maxSize,shell_minSize,shell_stdSize,weight,UMKP,AltRB,kPEX,Pivot_Br,S_Br
0,MANN-a81,20,3321,5506380,1658.0488,3317,3280,3280,1,3321.000000,3321,3321,0.000000,306.046574,1459.5,1800.0,1468.1,1477.4,1504.7
1,wikipedia_link_sr,20,1810,1637059,904.4525,1809,1798,1795,4,1806.250000,1810,1803,2.861381,66.902739,22.6,18.0,737.7,740.4,738.1
2,scc_reality,20,1332,880733,661.2110,1331,1235,1230,6,1290.333333,1332,1236,35.714921,47.684856,1800.0,1214.8,9.4,9.7,9.0
3,web-wikipedia_link_en13-all,20,432,93085,215.4745,431,427,427,1,432.000000,432,432,0.000000,184.662528,239.3,287.1,517.7,504.4,516.7
4,livejournal-links,20,442,94435,213.6538,441,371,369,3,412.333333,442,386,22.983086,0.269931,1.3,1.6,1.6,1.6,1.6
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
755,soc-flixster,2,264,11792,44.6667,150,67,52,16,215.000000,264,125,56.656862,0.784308,2.0,0.6,0.5,0.5,2.4
756,edit-ukwiki,2,281,14183,50.4733,280,65,40,26,235.384615,281,168,34.703222,0.437734,1.2,1.1,0.6,0.6,0.5
757,johnson8-4-4,2,70,1855,26.5000,53,53,53,1,70.000000,70,70,0.000000,0.389785,0.8,0.9,1.1,1.1,0.3
758,hamming6-2,2,64,1824,28.5000,57,57,57,1,64.000000,64,64,0.000000,0.081093,0.2,0.3,0.2,0.2,0.3


# Training with Random Forest



## Tie Breaking and Data Splitting

### **Balanced train-test split based on Y**

In [390]:
from data_splits import prepare_train_test_strat_Y

X_train, X_test, y_train, y_test, weights_train, weights_test = prepare_train_test_strat_Y(
    cat_df,
    tie_tolerance = 0.50  # algorithms within tie_tolerance% of min treated as ties
)

Expanded 687 rows with near ties using tolerance 50.0%.


In [391]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

# Train Random Forest with sample weights
rf_strat_y = RandomForestClassifier(
    n_estimators=400,
    max_depth=10,
    min_samples_split=2,
    class_weight='balanced',  # Handles class imbalance
    # random_state=42
)

rf_strat_y.fit(X_train, y_train, sample_weight=weights_train)



RandomForestClassifier(class_weight='balanced', max_depth=10, n_estimators=400)

Feature Importances

In [357]:

# Feature importance
importances = pd.DataFrame({
    'feature': X_train.columns,
    'importance': rf_strat_y.feature_importances_
}).sort_values('importance', ascending=False)

print("\nFeature Importances:")
print(importances)


Feature Importances:
          feature  importance
0         k_value    0.142944
3           d_avg    0.100475
7       shell_num    0.095317
6        core_min    0.092436
5        core_max    0.080921
11  shell_stdSize    0.080363
10  shell_minSize    0.073298
2               m    0.069479
4           d_max    0.069374
1               n    0.067769
8   shell_avgSize    0.066372
9   shell_maxSize    0.061253


#### Inference on 20% dataset

In [392]:
classification_metrics, time_metrics, predictions_df, wrong_predictions_df = kplex_dev.run_full_evaluation(
    cat_df=cat_df.loc[X_test.index], # for sending the values for test set only.
    clf=rf_strat_y,
    features=data_splits.feature_cols,
    algorithms=data_splits.algorithms,
    drop_indices= [425],
    epsilon=1e-3,
    save_files=True,               # set False if you don’t want to write files
    output_dir='./RF_results/tie_tol/50/expand_and_split',
    file_suffix= "inference_test_set_without_holdout_strat_y_exp_05" # change exp number
)

# Helper function: Write the displayed metrics to an excel file. Change number of sheet in accord with the experiment number
kplex_dev.write_metrics_to_excel_file(classification_metrics, time_metrics, 
                                    "./RF_results/tie_tol/50/expand_and_split/model_performance_for_all_experiments.xlsx", 
                                    'Inference_test_set_strat_y') # change exp number

Predicting on full dataset...

Classification performance:
Accuracy (counting ties as correct): 50.71%

Precision by Algorithm:
  UMKP: 23.08% (52 predictions)
  AltRB: 35.71% (56 predictions)
  kPEX: 67.65% (102 predictions)
  Pivot_Br: 45.86% (133 predictions)
  S_Br: 66.23% (77 predictions)

Recall by Algorithm:
  UMKP: 53.06% (26/49 cases)
  AltRB: 53.33% (32/60 cases)
  kPEX: 55.35% (119/215 cases)
  Pivot_Br: 69.68% (108/155 cases)
  S_Br: 55.70% (83/149 cases)

Dropped indices [425] before time‐metric evaluation.

Time performance metrics:
Time Metrics:
  Total Predicted Time = 54712.20
  Total Optimal Time   = 48282.00
  Time Ratio (pred/opt)= 1.133
  Avg Overhead/case ((pred - opt) / opt)    = 32.48%


Adding prediction probabilities...

Saving full predictions to './RF_results/tie_tol/50/expand_and_split\all_predictions_inference_test_set_without_holdout_strat_y_exp_05.csv'

Generating wrong-predictions with time metrics...

Saved 207 wrong predictions with time metrics

Top 

#### Re-evalute on entire dataset

In [282]:
classification_metrics, time_metrics, predictions_df, wrong_predictions_df = kplex_dev.run_full_evaluation(
    cat_df=cat_df,
    clf=rf_strat_y,
    features=data_splits.feature_cols,
    algorithms=data_splits.algorithms,
    drop_indices= [425],
    epsilon=1e-3,
    save_files=True,               # set False if you don’t want to write files
    output_dir='./RF_results/tie_tol/50/expand_and_split',
    file_suffix= "inference_entire_data_without_holdout_strat_y_exp_05" # change exp number
)

# Helper function: Write the displayed metrics to an excel file.
kplex_dev.write_metrics_to_excel_file(classification_metrics, time_metrics, 
                                    "./RF_results/tie_tol/50/expand_and_split/model_performance_for_all_experiments.xlsx", 
                                    'Inference_entire_data_strat_y') # change exp number

Predicting on full dataset...

Classification performance:
Accuracy (counting ties as correct): 55.53%

Precision by Algorithm:
  UMKP: 32.91% (79 predictions)
  AltRB: 46.49% (114 predictions)
  kPEX: 70.95% (210 predictions)
  Pivot_Br: 41.09% (202 predictions)
  S_Br: 71.61% (155 predictions)

Recall by Algorithm:
  UMKP: 54.00% (54/100 cases)
  AltRB: 65.52% (76/116 cases)
  kPEX: 60.06% (218/363 cases)
  Pivot_Br: 71.68% (162/226 cases)
  S_Br: 66.80% (171/256 cases)

Dropped indices [425] before time‐metric evaluation.

Time performance metrics:
Time Metrics:
  Total Predicted Time = 73574.30
  Total Optimal Time   = 66881.00
  Time Ratio (pred/opt)= 1.100
  Avg Overhead/case ((pred - opt) / opt)    = 19.76%


Adding prediction probabilities...

Saving full predictions to './RF_results/tie_tol/50/expand_and_split\all_predictions_inference_entire_data_without_holdout_strat_y_exp_05.csv'

Generating wrong-predictions with time metrics...

Saved 338 wrong predictions with time metri

The above results are shown when we drop the #row = 425

Epsilon is added to for cater #row = 425, where best algorithm ['AltRB'] has a runtime of 0 secs and predicted algorithm is Pivot_Br

In [87]:
## SHowing row 425

rt_opt  = cat_df[data_splits.algorithms].min(axis=1)
cat_df[rt_opt==0]

,dataset,k_value,n,m,d_avg,d_max,core_max,core_min,shell_num,shell_avgSize,shell_maxSize,shell_minSize,shell_stdSize,weight,UMKP,AltRB,kPEX,Pivot_Br,S_Br
425,hamming6-2,6,64,1824,28.5,57,57,57,1,64.0,64,64,0.0,0.0,19.9,0.0,6.7,6.8,11.2


### **Balanced train-test split based on both Y and k**

In [370]:
from data_splits import prepare_train_test_strat_Y_K

X_train, X_test, y_train, y_test, weights_train, weights_test = prepare_train_test_strat_Y_K(
    cat_df,
    tie_tolerance = 0.50  # algorithms within tie_tolerance% of min treated as ties
)

Expanded 524 rows with near ties using tolerance 10.0%.


In [371]:
# --- Model Training ---
rf_strat_y_K = RandomForestClassifier(
    n_estimators=400,
    max_depth=15,
    min_samples_split=2,
    class_weight='balanced',
    # random_state=42
)
rf_strat_y_K.fit(X_train, y_train, sample_weight=weights_train)

RandomForestClassifier(class_weight='balanced', max_depth=15, n_estimators=400)

Feature Importances

In [372]:
# Feature importance
importances = pd.DataFrame({
    'feature': X_train.columns,
    'importance': rf_strat_y_K.feature_importances_
}).sort_values('importance', ascending=False)

print("\nFeature Importances:")
print(importances)


Feature Importances:
          feature  importance
0         k_value    0.140751
3           d_avg    0.100289
7       shell_num    0.098944
6        core_min    0.094048
10  shell_minSize    0.086012
5        core_max    0.082130
4           d_max    0.077185
11  shell_stdSize    0.070233
2               m    0.066147
8   shell_avgSize    0.063990
1               n    0.061306
9   shell_maxSize    0.058963


#### Inference on 20% dataset

In [304]:
classification_metrics, time_metrics, predictions_df, wrong_predictions_df = kplex_dev.run_full_evaluation(
    cat_df=cat_df.loc[X_test.index], # for sending the values for test set only.
    clf=rf_strat_y_K,
    features=data_splits.feature_cols,
    algorithms=data_splits.algorithms,
    drop_indices= [425],
    epsilon=1e-3,
    save_files=True,               # set False if you don’t want to write files
    output_dir='./RF_results/tie_tol/50/expand_and_split',
    file_suffix= "inference_test_set_without_holdout_strat_y_K_exp_05" # change exp number
)

# Helper function: Write the displayed metrics to an excel file. Change number of sheet in accord with the experiment number
kplex_dev.write_metrics_to_excel_file(classification_metrics, time_metrics, 
                                    "./RF_results/tie_tol/50/expand_and_split/model_performance_for_all_experiments.xlsx", 
                                    'Inference_test_set_strat_y_K') # change exp number

Predicting on full dataset...

Classification performance:
Accuracy (counting ties as correct): 49.29%

Precision by Algorithm:
  UMKP: 34.88% (43 predictions)
  AltRB: 47.92% (48 predictions)
  kPEX: 65.52% (145 predictions)
  Pivot_Br: 37.50% (120 predictions)
  S_Br: 45.31% (64 predictions)

Recall by Algorithm:
  UMKP: 66.04% (35/53 cases)
  AltRB: 57.97% (40/69 cases)
  kPEX: 59.19% (132/223 cases)
  Pivot_Br: 72.90% (113/155 cases)
  S_Br: 56.85% (83/146 cases)

Dropped indices [425] before time‐metric evaluation.

Time performance metrics:
Time Metrics:
  Total Predicted Time = 54346.30
  Total Optimal Time   = 46404.40
  Time Ratio (pred/opt)= 1.171
  Avg Overhead/case ((pred - opt) / opt)    = 13.95%


Adding prediction probabilities...

Saving full predictions to './RF_results/tie_tol/50/expand_and_split\all_predictions_inference_test_set_without_holdout_strat_y_K_exp_05.csv'

Generating wrong-predictions with time metrics...

Saved 213 wrong predictions with time metrics

To

#### Re-evalute on entire dataset

In [305]:
classification_metrics, time_metrics, predictions_df, wrong_predictions_df = kplex_dev.run_full_evaluation(
    cat_df=cat_df,
    clf=rf_strat_y_K,
    features=data_splits.feature_cols,
    algorithms=data_splits.algorithms,
    drop_indices= [425],
    epsilon=1e-3,
    save_files=True,               # set False if you don’t want to write files
    output_dir='./RF_results/tie_tol/50/expand_and_split',
    file_suffix= "inference_entire_data_without_holdout_strat_y_K_exp_05" # change exp number
)

# Helper function: Write the displayed metrics to an excel file.
kplex_dev.write_metrics_to_excel_file(classification_metrics, time_metrics, 
                                    "./RF_results/tie_tol/50/expand_and_split/model_performance_for_all_experiments.xlsx", 
                                    'Inference_entire_data_strat_y_K') # change exp number

Predicting on full dataset...

Classification performance:
Accuracy (counting ties as correct): 53.68%

Precision by Algorithm:
  UMKP: 35.14% (74 predictions)
  AltRB: 49.57% (115 predictions)
  kPEX: 66.67% (252 predictions)
  Pivot_Br: 38.10% (189 predictions)
  S_Br: 65.38% (130 predictions)

Recall by Algorithm:
  UMKP: 55.00% (55/100 cases)
  AltRB: 62.93% (73/116 cases)
  kPEX: 62.26% (226/363 cases)
  Pivot_Br: 67.70% (153/226 cases)
  S_Br: 61.33% (157/256 cases)

Dropped indices [425] before time‐metric evaluation.

Time performance metrics:
Time Metrics:
  Total Predicted Time = 74797.40
  Total Optimal Time   = 66881.00
  Time Ratio (pred/opt)= 1.118
  Avg Overhead/case ((pred - opt) / opt)    = 22.19%


Adding prediction probabilities...

Saving full predictions to './RF_results/tie_tol/50/expand_and_split\all_predictions_inference_entire_data_without_holdout_strat_y_K_exp_05.csv'

Generating wrong-predictions with time metrics...

Saved 352 wrong predictions with time met

The above results are shown when we drop the #row = 425

Epsilon is added to for cater #row = 425, where best algorithm ['AltRB'] has a runtime of 0 secs and predicted algorithm is Pivot_Br

In [ ]:
## SHowing row 425

rt_opt  = results_df[algorithms].min(axis=1)
results_df[rt_opt==0][features+algorithms]

## Data Splitting and Tie Breaking 

First we split the data and then do the tie breaking.

Only use training set to produce multi-label aware training set with duplicate lines (but different labels)


In [373]:
feature_cols = data_splits.feature_cols
algorithms = data_splits.algorithms

### **Balanced train-test split based on Y**

In [374]:
train_df, test_df = data_splits.stratify_holdout_train_test_split(
    cat_df,
    target_col='target',
    test_size=0.2,
    random_state=40,
    # how='first'
)

Train set shape: (604, 20)
Test set shape:  (156, 20)


In [375]:
# Expanding rows withs ties, multi-label rows

import kplex_dev
import data_splits

X_train_, y_train_, weights_train_ = data_splits.expand_with_ties_and_prepare(
    train_df,
    weight_col='weight',
    target_col_name='target',
    tie_tolerance = 0.0 # algorithms within tie_tolerance% of min treated as ties
)


In [376]:
from sklearn.ensemble import RandomForestClassifier

# Train Random Forest with sample weights
rf_holdout_strat_y = RandomForestClassifier(
    n_estimators=200,
    max_depth=15,
    min_samples_split=2,
    class_weight='balanced',  # Handles class imbalance
    # random_state=42 
)

rf_holdout_strat_y.fit(X_train_, y_train_, sample_weight=weights_train_)


RandomForestClassifier(class_weight='balanced', max_depth=15, n_estimators=200)

Feature Importances

In [377]:
# Feature importance
importances = pd.DataFrame({
    'feature': X_train.columns,
    'importance': rf_holdout_strat_y.feature_importances_
}).sort_values('importance', ascending=False)

print("\nFeature Importances:")
print(importances)


Feature Importances:
          feature  importance
0         k_value    0.140837
6        core_min    0.121617
3           d_avg    0.117713
7       shell_num    0.095888
5        core_max    0.082980
10  shell_minSize    0.075583
11  shell_stdSize    0.072602
4           d_max    0.071473
2               m    0.062223
1               n    0.056762
8   shell_avgSize    0.051681
9   shell_maxSize    0.050640


#### Inference on 20% dataset

In [327]:
classification_metrics, time_metrics, predictions_df, wrong_predictions_df = kplex_dev.run_full_evaluation(
    cat_df=test_df, # for sending the values for test set only.
    clf=rf_holdout_strat_y,
    features=data_splits.feature_cols,
    algorithms=data_splits.algorithms,
    drop_indices= [425],
    epsilon=1e-3,
    save_files=True,               # set False if you don’t want to write files
    output_dir='./RF_results/tie_tol/50/split_and_expand',
    file_suffix= "inference_test_set_without_holdout_strat_y_exp_05" # change exp number
)

# Helper function: Write the displayed metrics to an excel file. Change number of sheet in accord with the experiment number
kplex_dev.write_metrics_to_excel_file(classification_metrics, time_metrics, 
                                    "./RF_results/tie_tol/50/split_and_expand/model_performance_for_all_experiments.xlsx", 
                                    'Inference_test_set_strat_y') # change exp number

Predicting on full dataset...

Classification performance:
Accuracy (counting ties as correct): 58.33%

Precision by Algorithm:
  UMKP: 30.00% (20 predictions)
  AltRB: 50.72% (69 predictions)
  kPEX: 68.37% (215 predictions)
  Pivot_Br: 31.48% (54 predictions)
  S_Br: 64.52% (62 predictions)

Recall by Algorithm:
  UMKP: 58.49% (31/53 cases)
  AltRB: 72.46% (50/69 cases)
  kPEX: 72.65% (162/223 cases)
  Pivot_Br: 64.52% (100/155 cases)
  S_Br: 62.33% (91/146 cases)

Dropped indices [425] before time‐metric evaluation.

Time performance metrics:
Time Metrics:
  Total Predicted Time = 52170.10
  Total Optimal Time   = 46404.40
  Time Ratio (pred/opt)= 1.124
  Avg Overhead/case ((pred - opt) / opt)    = 141.53%


Adding prediction probabilities...

Saving full predictions to './RF_results/tie_tol/50/split_and_expand\all_predictions_inference_test_set_without_holdout_strat_y_exp_05.csv'

Generating wrong-predictions with time metrics...

Saved 175 wrong predictions with time metrics

Top 

#### Re-evalute on entire dataset

In [326]:
classification_metrics, time_metrics, predictions_df, wrong_predictions_df = kplex_dev.run_full_evaluation(
    cat_df=cat_df,
    clf=rf_holdout_strat_y,
    features=data_splits.feature_cols,
    algorithms=data_splits.algorithms,
    drop_indices= [425],
    epsilon=1e-3,
    save_files=True,               # set False if you don’t want to write files
    output_dir='./RF_results/tie_tol/50/split_and_expand',
    file_suffix= "inference_entire_data_without_holdout_strat_y_exp_05" # change exp number
)

# Helper function: Write the displayed metrics to an excel file.
kplex_dev.write_metrics_to_excel_file(classification_metrics, time_metrics, 
                                    "./RF_results/tie_tol/50/split_and_expand/model_performance_for_all_experiments.xlsx", 
                                    'Inference_entire_data_strat_y') # change exp number

Predicting on full dataset...

Classification performance:
Accuracy (counting ties as correct): 57.63%

Precision by Algorithm:
  UMKP: 44.74% (38 predictions)
  AltRB: 52.24% (134 predictions)
  kPEX: 65.93% (361 predictions)
  Pivot_Br: 28.44% (109 predictions)
  S_Br: 69.49% (118 predictions)

Recall by Algorithm:
  UMKP: 47.00% (47/100 cases)
  AltRB: 74.14% (86/116 cases)
  kPEX: 71.35% (259/363 cases)
  Pivot_Br: 61.95% (140/226 cases)
  S_Br: 59.77% (153/256 cases)

Dropped indices [425] before time‐metric evaluation.

Time performance metrics:
Time Metrics:
  Total Predicted Time = 81200.50
  Total Optimal Time   = 66881.00
  Time Ratio (pred/opt)= 1.214
  Avg Overhead/case ((pred - opt) / opt)    = 205.53%


Adding prediction probabilities...

Saving full predictions to './RF_results/tie_tol/50/split_and_expand\all_predictions_inference_entire_data_without_holdout_strat_y_exp_05.csv'

Generating wrong-predictions with time metrics...

Saved 322 wrong predictions with time metr

The above results are shown when we drop the #row = 425

Epsilon is added to for cater #row = 425, where best algorithm ['AltRB'] has a runtime of 0 secs and predicted algorithm is Pivot_Br

### **Balanced train-test split based on Y and K**

In [380]:
train_df, test_df = data_splits.stratify_holdout_train_test_split(
    cat_df,
    target_col='target',
    k_value_col='k_value',
    test_size=0.2,
    random_state=40,
    # how='first' # or 'concat'
)

Train set shape: (612, 20)
Test set shape:  (148, 20)


In [381]:
import kplex_dev
import data_splits

X_train_, y_train_, weights_train_ = data_splits.expand_with_ties_and_prepare(
    train_df,
    weight_col='weight',
    target_col_name='target',
    tie_tolerance = 0.0 # algorithms within tie_tolerance% of min treated as ties

)


In [382]:
from sklearn.ensemble import RandomForestClassifier

# Train Random Forest with sample weights
rf_holdout_strat_y_K = RandomForestClassifier(
    n_estimators=300,
    max_depth=10,
    min_samples_split=2,
    class_weight='balanced',  # Handles class imbalance
    # random_state=42  best
)

rf_holdout_strat_y_K.fit(X_train_, y_train_, sample_weight=weights_train_)


RandomForestClassifier(class_weight='balanced', max_depth=18, n_estimators=400)

Feature Importances

In [383]:
# Feature importance
importances = pd.DataFrame({
    'feature': X_train.columns,
    'importance': rf_holdout_strat_y_K.feature_importances_
}).sort_values('importance', ascending=False)

print("\nFeature Importances:")
print(importances)


Feature Importances:
          feature  importance
0         k_value    0.117118
7       shell_num    0.108458
6        core_min    0.106523
3           d_avg    0.104605
4           d_max    0.091314
5        core_max    0.082720
11  shell_stdSize    0.075183
2               m    0.074436
10  shell_minSize    0.074176
9   shell_maxSize    0.058102
8   shell_avgSize    0.055885
1               n    0.051480


#### Inference on 20% dataset

In [348]:
classification_metrics, time_metrics, predictions_df, wrong_predictions_df = kplex_dev.run_full_evaluation(
    cat_df=test_df, # for sending the values for test set only.
    clf=rf_holdout_strat_y_K,
    features=data_splits.feature_cols,
    algorithms=data_splits.algorithms,
    drop_indices= [425],
    epsilon=1e-3,
    save_files=True,               # set False if you don’t want to write files
    output_dir='./RF_results/tie_tol/50/split_and_expand',
    file_suffix= "inference_test_set_without_holdout_strat_y_K_exp_05" # change exp number
)

# Helper function: Write the displayed metrics to an excel file. Change number of sheet in accord with the experiment number
kplex_dev.write_metrics_to_excel_file(classification_metrics, time_metrics, 
                                    "./RF_results/tie_tol/50/split_and_expand/model_performance_for_all_experiments.xlsx", 
                                    'Inference_test_set_strat_y_K') # change exp number

Predicting on full dataset...

Classification performance:
Accuracy (counting ties as correct): 56.19%

Precision by Algorithm:
  UMKP: 20.00% (45 predictions)
  AltRB: 43.33% (60 predictions)
  kPEX: 73.72% (137 predictions)
  Pivot_Br: 46.38% (69 predictions)
  S_Br: 62.39% (109 predictions)

Recall by Algorithm:
  UMKP: 56.60% (30/53 cases)
  AltRB: 66.67% (46/69 cases)
  kPEX: 62.78% (140/223 cases)
  Pivot_Br: 65.16% (101/155 cases)
  S_Br: 73.97% (108/146 cases)

Dropped indices [425] before time‐metric evaluation.

Time performance metrics:
Time Metrics:
  Total Predicted Time = 52508.30
  Total Optimal Time   = 46404.40
  Time Ratio (pred/opt)= 1.132
  Avg Overhead/case ((pred - opt) / opt)    = 10.67%


Adding prediction probabilities...

Saving full predictions to './RF_results/tie_tol/50/split_and_expand\all_predictions_inference_test_set_without_holdout_strat_y_K_exp_05.csv'

Generating wrong-predictions with time metrics...

Saved 184 wrong predictions with time metrics

T

#### Re-evalute on entire dataset

In [349]:
classification_metrics, time_metrics, predictions_df, wrong_predictions_df = kplex_dev.run_full_evaluation(
    cat_df=cat_df,
    clf=rf_holdout_strat_y_K,
    features=data_splits.feature_cols,
    algorithms=data_splits.algorithms,
    drop_indices= [425],
    epsilon=1e-3,
    save_files=True,               # set False if you don’t want to write files
    output_dir='./RF_results/tie_tol/50/split_and_expand',
    file_suffix= "inference_entire_data_without_holdout_strat_y_K_exp_05" # change exp number
)

# Helper function: Write the displayed metrics to an excel file.
kplex_dev.write_metrics_to_excel_file(classification_metrics, time_metrics, 
                                    "./RF_results/tie_tol/50/split_and_expand/model_performance_for_all_experiments.xlsx", 
                                    'Inference_entire_data_strat_y_K') # change exp number

Predicting on full dataset...

Classification performance:
Accuracy (counting ties as correct): 57.89%

Precision by Algorithm:
  UMKP: 27.85% (79 predictions)
  AltRB: 41.74% (115 predictions)
  kPEX: 74.37% (238 predictions)
  Pivot_Br: 45.11% (133 predictions)
  S_Br: 68.21% (195 predictions)

Recall by Algorithm:
  UMKP: 45.00% (45/100 cases)
  AltRB: 63.79% (74/116 cases)
  kPEX: 64.74% (235/363 cases)
  Pivot_Br: 65.49% (148/226 cases)
  S_Br: 74.61% (191/256 cases)

Dropped indices [425] before time‐metric evaluation.

Time performance metrics:
Time Metrics:
  Total Predicted Time = 74434.80
  Total Optimal Time   = 66881.00
  Time Ratio (pred/opt)= 1.113
  Avg Overhead/case ((pred - opt) / opt)    = 20.24%


Adding prediction probabilities...

Saving full predictions to './RF_results/tie_tol/50/split_and_expand\all_predictions_inference_entire_data_without_holdout_strat_y_K_exp_05.csv'

Generating wrong-predictions with time metrics...

Saved 320 wrong predictions with time met

The above results are shown when we drop the #row = 425

Epsilon is added to for cater #row = 425, where best algorithm ['AltRB'] has a runtime of 0 secs and predicted algorithm is Pivot_Br